In [ ]:
import csv
import json
from neo4j import GraphDatabase

# -------------------------
# Neo4j connection settings
# -------------------------
with open('../neo4j_dbinfo', 'r') as f:
    neo4j_info = json.load(f)

NEO4J_URI = neo4j_info["uri"]
NEO4J_USER = neo4j_info["username"]
NEO4J_PASSWORD = neo4j_info["password"]
BATCH_SIZE = 10000

## ***1. Creating CTD GENE node first***
- create first connect later
- because CTD provides 1 file for 1 table, we can just create node without merge
- We can also keep coherence unless we don't interrupt this script

In [ ]:
import csv
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def import_ctd_batch(tx, rows):
    query = """
    UNWIND $rows AS row
    CREATE (r:Gene {
        ChemicalName: row.ChemicalName,
        ChemicalID: row.ChemicalID,
        CasRN: row.CasRN,
        GeneSymbol: row.GeneSymbol,
        GeneID: row.GeneID,
        GeneForms: row.GeneForms,
        Organism: row.Organism,
        OrganismID: row.OrganismID,
        Interaction: row.Interaction,
        InteractionActions: row.InteractionActions,
        PubMedIDs: row.PubMedIDs
    })
    """
    tx.run(query, rows=rows)

with driver.session() as session, open(CSV_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    batch = []
    for i, row in enumerate(reader, start=1):
        batch.append(row)
        if len(batch) >= BATCH_SIZE:  # commit every 1000 rows
            session.write_transaction(import_ctd_batch, batch)
            print(f"Inserted {i} rows...")
            batch = []
    if batch:
        session.write_transaction(import_ctd_batch, batch)
        print(f"Inserted {i} rows (final batch).")

driver.close()
print("CTD interactions imported successfully with all columns as properties on Gene nodes.")


### ***Connecting edge from Gene to Chemical node later***

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def create_relationships_batch(tx, batch):
    query = """
    UNWIND $batch AS c
    MATCH (chem:Chemical {casrn: c.casrn})
    MATCH (go:CTD_Chem_Gene {CasRN: c.casrn})
    MERGE (chem)-[:AFFECTS]->(go)
    """
    tx.run(query, batch=batch)

with driver.session() as session:
    offset = 0
    while True:
        # Fetch batch of Chemical nodes
        result = session.run(
            f"""
            MATCH (c:Chemical)
            RETURN c.casrn AS casrn
            SKIP {offset} LIMIT {BATCH_SIZE}
            """
        )
        batch = [{"casrn": record["casrn"]} for record in result]

        if not batch:
            break  # no more nodes

        session.write_transaction(create_relationships_batch, batch)
        offset += BATCH_SIZE
        print(f"Processed {offset} Chemical nodes...")

driver.close()
print("Bulk Chemical -> Gene relationships created.")

### ***2. Creating GO node of CTD first***

In [ ]:
import csv
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Path to your CSV file
csv_file = "CTD_chem_go_enriched.csv"

def create_node(tx, row):
    query = """
    CREATE (n:CTD_Chem_GO)
    SET n.ChemicalName = $ChemicalName,
        n.ChemicalID = $ChemicalID,
        n.CasRN = $CasRN,
        n.Ontology = $Ontology,
        n.GOTermName = $GOTermName,
        n.GOTermID = $GOTermID,
        n.HighestGOLevel = $HighestGOLevel,
        n.PValue = $PValue,
        n.CorrectedPValue = $CorrectedPValue,
        n.TargetMatchQty = $TargetMatchQty,
        n.TargetTotalQty = $TargetTotalQty,
        n.BackgroundMatchQty = $BackgroundMatchQty,
        n.BackgroundTotalQty = $BackgroundTotalQty,
        n.name = $GOTermName
    """
    tx.run(query, **row)

# Open CSV and create nodes
with driver.session() as session:
    with open(csv_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Convert numeric fields
            row["HighestGOLevel"] = int(row["HighestGOLevel"])
            row["PValue"] = float(row["PValue"])
            row["CorrectedPValue"] = float(row["CorrectedPValue"])
            row["TargetMatchQty"] = int(row["TargetMatchQty"])
            row["TargetTotalQty"] = int(row["TargetTotalQty"])
            row["BackgroundMatchQty"] = int(row["BackgroundMatchQty"])
            row["BackgroundTotalQty"] = int(row["BackgroundTotalQty"])
            
            session.write_transaction(create_node, row)

driver.close()
print("All nodes created successfully!")


### ***Connecting edge to Chemical node***

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def create_relationships_batch(tx, batch):
    query = """
    UNWIND $batch AS c
    MATCH (chem:Chemical {casrn: c.casrn})
    MATCH (go:CTD_Chem_GO {CasRN: c.casrn})
    MERGE (chem)-[:AFFECTS]->(go)
    """
    tx.run(query, batch=batch)

with driver.session() as session:
    offset = 0
    while True:
        # Fetch batch of Chemical nodes
        result = session.run(
            f"""
            MATCH (c:Chemical)
            RETURN c.casrn AS casrn
            SKIP {offset} LIMIT {BATCH_SIZE}
            """
        )
        batch = [{"casrn": record["casrn"]} for record in result]

        if not batch:
            break  # no more nodes

        session.write_transaction(create_relationships_batch, batch)
        offset += BATCH_SIZE
        print(f"Processed {offset} Chemical nodes...")

driver.close()
print("Bulk Chemical -> GO relationships created.")

### ***Creating Disease node of CTD first***

In [ ]:
import csv
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Path to your CSV file
csv_file = "CTD_chemicals_diseases.csv"

def create_nodes(tx, rows):
    query = """
    UNWIND $rows AS row
    CREATE (n:CTD_Chem_Disease)
    SET n.ChemicalName = row.ChemicalName,
        n.ChemicalID = row.ChemicalID,
        n.CasRN =row.CasRN,
        n.DiseaseName = row.DiseaseName,
        n.DiseaseID = row.DiseaseID,
        n.DirectEvidence = row.DirectEvidence,
        n.InferenceGeneSymbol = row.InferenceGeneSymbol,
        n.InferenceScore = row.InferenceScore,
        n.OmimIDs = row.OmimIDs,
        n.PubMedIDs = row.PubMedIDs,
        n.name = row.DiseaseName
    """
    tx.run(query, rows=rows)


with driver.session() as session, open(csv_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    batch = []
    for i, row in enumerate(reader, start=1):
        batch.append(row)
        if len(batch) >= BATCH_SIZE:  # commit every 1000 rows
            session.execute_write(create_nodes, batch)
            print(f"Inserted {i} rows...")
            batch = []
    if batch:
        session.execute_write(create_nodes, batch)
        print(f"Inserted {i} rows (final batch).")
            

driver.close()
print("All nodes created successfully!")


### ***Connecting edge to Chemical node later***

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
BATCH_SIZE = 100
def create_relationships_batch(tx, batch):
    query = """
    UNWIND $batch AS c
    MATCH (chem:Chemical {casrn: c.casrn})
    MATCH (disease:CTD_Chem_Disease {CasRN: c.casrn})
    MERGE (chem)-[:AFFECTS]->(disease)
    """
    tx.run(query, batch=batch)

with driver.session() as session:
    offset = 0
    while True:
        # Fetch batch of Chemical nodes
        result = session.run(
            f"""
            MATCH (c:Chemical)
            RETURN c.casrn AS casrn
            SKIP {offset} LIMIT {BATCH_SIZE}
            """
        )
        batch = [{"casrn": record["casrn"]} for record in result]

        if not batch:
            break  # no more nodes

        session.execute_write(create_relationships_batch, batch)
        offset += BATCH_SIZE
        print(f"Processed {offset} Chemical nodes...")

driver.close()
print("Bulk Chemical -> Disease relationships created.")